I denna Notebook skapar och testar jag min RAG-modell. Jag har sedan byggt appen som skall använda min RAG-modell i en separat fil (app.py).

Jag har valt att använda OpenAIs LLM för att min chatbot skall kunna kommunicera "fritt", och använder i detta dokument FAISS som vektordatabas. Jag använde först Chromadb men stötte på trubbel när jag kopplade min RAG-modell till min Streamlit-app pga olika versionskonflikter.

För att börja smått och se om jag kan få något som fungerar börjar jag med att bara spara 3 textfiler i mappen för torr-hud. Om det fungerar som jag tänkt så skapar jag fler mappar för andra hudtyper.

(När ovan fungerade uppdaterade jag nedan kod till att läsa in filer för även kombinerad samt fet hud).

Längst ner finns ytterligare kommentarer och reflektioner.

In [30]:
# En kod för att rensa tidigare inläsningar om något skall uppdateras, för att undvika att det krockar med gammal data
base_folder = "." # Hänvisar till den lokala mappen
all_documents = [] # Skapar en tom lista med alla dokument som laddas in

In [3]:
# Alla imports som behövs för att köra samtliga kodblock
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import pandas as pd

Börjar med att läsa in alla filer och använder filnamnen för att dela in filerna i olika kategorier och hudtyper, och använder dessa filer för att skapa chunks.

In [33]:

# Skapar en tom behållare för att samla in all textdata innan den bearbetas
base_folder = "." 
all_documents = [] 

# Ladda in alla txt-filer och används filnamnen för att tagga filerna så att RAG enkelt kan hitta
for root, dirs, files in os.walk(base_folder):
    for filename in files:
        if filename.endswith(".txt"):
            file_path = os.path.join(root, filename) # här skapas sökväg baserat på filnamn
            
            try: # Testar att läsa in filer med UTF-8-kodning, och hoppar vidare till nästa fil om någon fil är skadad utan att krasha
                loader = TextLoader(file_path, encoding='utf-8')
                loaded_docs = loader.load()
                
                for doc in loaded_docs: # Skapar variabler med små bokstäver så att inga ord missas pga stora/små bokstäver i filnamnet
                    file_lower = filename.lower()
                    path_lower = file_path.lower()
                    
                    # Delar in i olika hudtyper och sparar som metadata utifrån filnamn
                    if "torr-hud" in path_lower or "torr-hud" in file_lower:
                        doc.metadata["skin_type"] = "torr-hud"
                    elif "fet-hud" in path_lower or "fet-hud" in file_lower:
                        doc.metadata["skin_type"] = "fet-hud"
                    elif "kombinerad-hud" in path_lower or "kombinerad-hud" in file_lower:
                        doc.metadata["skin_type"] = "kombinerad-hud"
                    else:
                        doc.metadata["skin_type"] = "allmänt"
                    
                    # Delar upp i kategorier ch sparar som metadata utifrån filnamn
                    if "rengöring" in file_lower or "rengoring" in file_lower:
                        doc.metadata["category"] = "rengöring"
                    elif "serum" in file_lower:
                        doc.metadata["category"] = "serum"
                    elif "ansiktskräm" in file_lower or "ansiktskram" in file_lower or "lotion" in file_lower:
                        doc.metadata["category"] = "ansiktskräm"
                    else:
                        doc.metadata["category"] = "allmänt"
                    
                    # Sparar källan för spårbarhet
                    doc.metadata["source"] = filename
                    all_documents.append(doc)
            except Exception as e:
                print(f"Kunde inte ladda {filename}: {e}") # Felhantering om filen inte kan laddas in och sparas enligt ovan

# Delar upp txt-dokumenten i chunks. Recurive delar upp texten vid naturliga tecken som radbrytningar eller punkter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_chunks = text_splitter.split_documents(all_documents)

print(f"KLART! Totalt antal chunks skapade: {len(final_chunks)}")

KLART! Totalt antal chunks skapade: 31


Ovan ser korrekt ut med 31 skapade chunks. Nu går jag vidare med att skapa embeddings som "fångar" kontexten av alla chunks genom att omvandla alla textstycken (chunks) till matematiska vektorer (listor med siffror).

In [34]:
# Initierar embeddings och använder nyckel mot OpenAIs Embedding modell
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY") # Kodblocket med själva koden är borttagen då den inte får laddas upp på Git (har sparat den separat)
)

# Skapar ett FAISS-index av alla chunks
vectorstore = FAISS.from_documents(
    documents=final_chunks,
    embedding=embeddings
)

# Sparar index lokalt
vectorstore.save_local("faiss_skincare_v3")

print(f"Nytt FAISS-index skapat med {len(final_chunks)} taggade textbitar!")

# Testa att söka efter rengöring för fet hud
filtered_results = vectorstore.similarity_search(
    "Vilken tvätt passar mig?",
    k=10,
    filter={
        "$and": [
            {"skin_type": "fet-hud"},
            {"category": "rengöring"}
        ]
    }
)

print("\n--- Resultat av test ---")
for r in filtered_results[:2]:
    print(f"Källa: {r.metadata['source']}")
    print(f"Innehåll: {r.page_content[:150]}...")
    print("-" * 10)

Nytt FAISS-index skapat med 31 taggade textbitar!

--- Resultat av test ---
Källa: rengöring-fet-hud.txt
Innehåll: Pris: 315 kr.

Länk: https://www.kicks.se/kiehls-rare-earth-deep-pore-daily-cleanser-150-ml

2. COSRX - Low pH Good Morning Gel Cleanser

Typ: Mild re...
----------
Källa: rengöring-fet-hud.txt
Innehåll: Tips på ansiktsrengöring för fet hy

För en fet hudtyp behöver vi rengöringar som effektivt löser upp överskottsfett och rengör porerna på djupet utan...
----------


Ovan test fungerade bra då jag fick rätt rekommendation från rätt txt-fil! Går vidare med att skapa LLM-kedjan som pratar med databasen och sammanfattar svaren åt användaren för att skapa ett bättre test och se hur chatboten svarar.

In [35]:
# TEST -- Här testade jag flera olika questions, skin_type och category för att se olika svar från modellen
user_question = "Jag vill ha hjälp med hudvård."
selected_skin_type = "fet-hud"
selected_category = "ansiktskräm"

# Kopplar upp till LLM
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

# Justerade prompt då jag först fick upprepade svar
system_prompt = (
    "Du är en hudvårdsexpert på Kicks. "
    "Använd endast informationen i kontexten för att svara på frågan. "
    "VIKTIGT: Om samma produkt förekommer flera gånger i kontexten, presentera den bara EN gång. "
    "Om ingen produkt matchar filtren exakt, säg det tydligt och föreslå då något närliggande alternativ från kontexten. "
    "Hitta inte på produkter, priser eller länkar."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

vectorstore = FAISS.load_local(
    "faiss_skincare_v3", # Se till att det är rätt vektordatabas här
    embeddings,
    allow_dangerous_deserialization=True
)

# Hämta topp 20 kandidater först
candidate_docs = vectorstore.similarity_search(
    user_question,
    k=20
)

# Filtrera strikt efter användarens hudtyp och produkt
strict_docs = [
    doc for doc in candidate_docs
    if doc.metadata.get("skin_type") == selected_skin_type
    and doc.metadata.get("category") == selected_category
]

# Om inget exakt matchar: försök med bara hudtyp
fallback_docs = [
    doc for doc in candidate_docs
    if doc.metadata.get("skin_type") == selected_skin_type
]

if strict_docs:
    final_docs = strict_docs[:5]
    mode_text = f"Exakt filtermatch: {selected_skin_type} + {selected_category}"
elif fallback_docs:
    final_docs = fallback_docs[:5]
    mode_text = f"Ingen exakt {selected_category}; fallback till {selected_skin_type}"
else:
    final_docs = []
    mode_text = "Ingen relevant kontext hittades"

context = "\n\n".join(doc.page_content for doc in final_docs)

messages = prompt.invoke({
    "input": user_question,
    "context": context if context else "Ingen relevant kontext hittades."
})

response = llm.invoke(messages)

print("\n--- TEST-INFO ---")
print(mode_text)
print(f"Antal träffar efter manuell filtrering: {len(final_docs)}")

print("\n--- HÄMTADE DOKUMENT ---")
for i, doc in enumerate(final_docs, 1):
    print(f"\nDokument {i}")
    print("Metadata:", doc.metadata)
    print("Innehåll:", doc.page_content[:250])

print("\n--- CHATBOTENS SVAR ---")
print(response.content)


--- TEST-INFO ---
Exakt filtermatch: fet-hud + ansiktskräm
Antal träffar efter manuell filtrering: 2

--- HÄMTADE DOKUMENT ---

Dokument 1
Metadata: {'source': 'ansiktskräm-fet-hud.txt', 'skin_type': 'fet-hud', 'category': 'ansiktskräm'}
Innehåll: 2. La Roche-Posay - Effaclar Mat

Typ: Mattgörande ansiktskräm.

Varför den passar: Motverkar glans och hjälper till att dra ihop porerna. Ger en matt finish som håller hela dagen.

Pris: 235 kr.

Länk: https://www.kicks.se/la-roche-posay-effaclar-ma

Dokument 2
Metadata: {'source': 'ansiktskräm-fet-hud.txt', 'skin_type': 'fet-hud', 'category': 'ansiktskräm'}
Innehåll: Tips på ansiktskräm för fet hy

För fet hud är målet att återfukta utan att täppa till porerna eller addera onödig olja. Lätta gel-konsistenser är oftast bäst.

1. Clinique - Dramatically Different Oil-Free Gel

Typ: Oljefri fuktighetsgel.

Varför de

--- CHATBOTENS SVAR ---
Självklart! Om du letar efter en ansiktskräm för fet hy, kan jag rekommendera följande produkter:

1. *

Jag är nöjd med ovan output då den ger rätt rekommendation och ett trevligt och "mänskligt" svar. Nästa steg är att göra ytterligare ett test för att utvärdera modellens svar och säkerställa att det blir rätt.



In [ ]:
# ===== UTVÄRDERING AV RAG-MODELL =====


# 1. LLM för att GENERERA svar från din RAG-modell
answer_llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

# 2. LLM för att BEDÖMA svaret
judge_llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0
)

# 3. Ladda embeddings + FAISS
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

vectorstore = FAISS.load_local(
    "faiss_skincare_v3",
    embeddings,
    allow_dangerous_deserialization=True
)

# 4. Samma promptidé som i ditt test
system_prompt = (
    "Du är en hudvårdsexpert på Kicks. "
    "Använd endast informationen i kontexten för att svara på frågan. "
    "VIKTIGT: Om samma produkt förekommer flera gånger i kontexten, presentera den bara EN gång. "
    "Om ingen produkt matchar filtren exakt, säg det tydligt och föreslå då något närliggande alternativ från kontexten. "
    "Inkludera pris och länk till produkter du rekommenderar. "
)

# 5. Testfrågor med facit
# Här kan man variera frågor och korrekta svar 
test_cases = [
    {
        "question": "Jag har fet hud och vill ha en ansiktskräm. Vad rekommenderar du?",
        "skin_type": "fet-hud",
        "category": "ansiktskräm",
        "correct_answer": "Svaret ska nämna att återfuktning är viktigt utan att täppa till porerna eller addera onödig olja. Svaret skall inkludera minst en produkt från kontexten, inkl pris och länk. Om ingen exakt match finns ska svaret tydligt säga det, utan att hitta på produkter eller länkar."
    },
    {
        "question": "Jag har torr hud och söker rengöring. Vad passar mig?",
        "skin_type": "torr-hud",
        "category": "rengöring",
        "correct_answer": "Svaret ska rekommendera att torr hy använder milda rengöringar. Svaret skall inkludera minst en produkt från kontexten, inkl pris och länk. Om ingen exakt match finns ska svaret tydligt säga det, utan att hitta på produkter eller länkar."
    },
    {
        "question": "Jag har kombinerad hud och vill ha serum. Har du något tips?",
        "skin_type": "kombinerad-hud",
        "category": "serum",
        "correct_answer": "Svaret ska nämna att serum för kombinerad hud bör ge mycket fukt (hyaluronsyra). Svaret skall inkludera minst en produkt från kontexten, inkl pris och länk. Om ingen exakt match finns ska svaret tydligt säga det, utan att hitta på produkter eller länkar."
    }
]

def run_rag(question, selected_skin_type, selected_category, k=20):
    # Hämta kandidater
    candidate_docs = vectorstore.similarity_search(question, k=k)

    # Strikt filtrering
    strict_docs = [
        doc for doc in candidate_docs
        if doc.metadata.get("skin_type") == selected_skin_type
        and doc.metadata.get("category") == selected_category
    ]

    # Fallback: bara hudtyp
    fallback_docs = [
        doc for doc in candidate_docs
        if doc.metadata.get("skin_type") == selected_skin_type
    ]

    if strict_docs:
        final_docs = strict_docs[:5]
        mode_text = f"Exakt filtermatch: {selected_skin_type} + {selected_category}"
    elif fallback_docs:
        final_docs = fallback_docs[:5]
        mode_text = f"Ingen exakt {selected_category}; fallback till {selected_skin_type}"
    else:
        final_docs = []
        mode_text = "Ingen relevant kontext hittades"

    context = "\n\n".join(doc.page_content for doc in final_docs)

    user_prompt = f"""
Fråga: {question}

Kontext:
{context if context else "Ingen relevant kontext hittades."}
"""

    messages = [
        ("system", system_prompt),
        ("human", user_prompt)
    ]

    response = answer_llm.invoke(messages)

    return {
        "answer": response.content,
        "context": context,
        "mode_text": mode_text,
        "num_docs": len(final_docs)
    }

def grade_answer(question, correct_answer, student_answer):
    judge_prompt = f"""
Du är en strikt men rättvis utvärderare.

Du ska jämföra modellens svar med facit och sätta ett betyg från 1 till 5 enligt denna skala:

1 = Svaret ligger långt ifrån det rätta svaret
2 = Svaret ligger fortfarande långt ifrån, men innehåller ändå relevant information
3 = Delar av rätta svaret finns med, men en del missas
4 = Ett bra svar! Mycket rätt information finns med, även om kanske några meningar saknas
5 = Högsta betyg! Svaret stämmer med det rätta svaret, och all relevant information finns med. Det är okej om kanske en mening saknas

Viktigt:
- Bedöm endast innehållet i svaret.
- Var extra uppmärksam på om svaret håller sig till facit.
- Ge endast betyg 1, 2, 3, 4 eller 5 på första raden.
- Ge sedan en kort motivering på svenska på andra raden.

Fråga:
{question}

Facit:
{correct_answer}

Modellens svar:
{student_answer}
"""

    evaluation = judge_llm.invoke(judge_prompt).content.strip()

    lines = evaluation.split("\n")
    score_line = lines[0].strip()

    try:
        score = int(score_line[0])
        if score not in [1, 2, 3, 4, 5]:
            score = None
    except:
        score = None

    motivation = "\n".join(lines[1:]).strip() if len(lines) > 1 else evaluation

    return score, motivation

# 6. Kör alla tester
results = []

for i, case in enumerate(test_cases, 1):
    rag_result = run_rag(
        question=case["question"],
        selected_skin_type=case["skin_type"],
        selected_category=case["category"]
    )

    score, motivation = grade_answer(
        question=case["question"],
        correct_answer=case["correct_answer"],
        student_answer=rag_result["answer"]
    )

    results.append({
        "Test": i,
        "Fråga": case["question"],
        "Hudtyp": case["skin_type"],
        "Kategori": case["category"],
        "Betyg": score,
        "Motivering": motivation,
        "Modellsvar": rag_result["answer"]
    })

# # 7. Visa resultat

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

df_results = pd.DataFrame(results)

# Ändra ordning på kolumnerna så att betyget kommer först
df_results = df_results[
    [
        "Betyg",
        "Test",
        "Fråga",
        "Hudtyp",
        "Kategori",
        "Motivering",
        "Modellsvar"
    ]
]

display(df_results)

valid_scores = [r["Betyg"] for r in results if isinstance(r["Betyg"], int)]
if valid_scores:
    print(f"Medelbetyg: {sum(valid_scores) / len(valid_scores):.2f} / 5")
else:
    print("Kunde inte räkna ut medelbetyg.")

,Betyg,Test,Fråga,Hudtyp,Kategori,Motivering,Modellsvar
0,5,1,Jag har fet hud och vill ha en ansiktskräm. Vad rekommenderar du?,fet-hud,ansiktskräm,"Svaret innehåller all relevant information, inklusive rekommendationer för produkter som återfuktar utan att täppa till porerna, samt pris och länkar.","För dig med fet hud rekommenderar jag följande ansiktskrämer som kommer att återfukta utan att göra huden glansig:\n\n1. **Clinique - Dramatically Different Oil-Free Gel**\n\n - Typ: Oljefri fuktighetsgel\n - Varför den passar: En lätt, oljefri formula som balanserar huden och ger precis rätt mängd fukt utan att huden känns fet eller glansig.\n - Pris: 409 kr\n - [Köp här](https://www.kicks.se/clinique-dramatically-different-gel-skin-type-34-125-ml)\n\n2. **La Roche-Posay - Effaclar Mat**\n\n - Typ: Mattgörande ansiktskräm\n - Varför den passar: Motverkar glans och hjälper till att dra ihop porerna. Ger en matt finish som håller hela dagen.\n - Pris: 235 kr\n - [Köp här](https://www.kicks.se/la-roche-posay-effaclar-mat-40-ml)\n\nBåda alternativen är utmärkta för fet hud och hjälper till att kontrollera glans och hålla huden balanserad."
1,4,2,Jag har torr hud och söker rengöring. Vad passar mig?,torr-hud,rengöring,"Svaret ger en bra rekommendation för torr hud med milda rengöringar och inkluderar produkter med pris och länk, men det är oklart om dessa produkter fanns i den givna kontexten.",För torr hud rekommenderar jag följande rengöringsprodukter som är särskilt anpassade för att rengöra utan att torka ut din hud:\n\n1. **CeraVe - Hydrating Cleanser**\n - **Typ:** Återfuktande rengöringslotion.\n - **Varför den passar:** Innehåller tre essentiella ceramider och hyaluronsyra som både rengör och hjälper till att återställa hudens skyddsbarriär utan att löddra.\n - **Pris:** 109 kr.\n - **Länk:** [CeraVe Hydrating Cleanser](https://www.kicks.se/cerave-hydrating-cleanser-236-ml)\n\n2. **Clinique - Take The Day Off Cleansing Balm**\n - **Typ:** Rengöringsbalm.\n - **Varför den passar:** Förvandlas från ett fast balm till en silkeslen olja vid applicering. Löser effektivt upp makeup och solskydd samtidigt som den lämnar huden mjuk och genomfuktad.\n - **Pris:** 429 kr.\n - **Länk:** [Clinique Take The Day Off Cleansing Balm](https://www.kicks.se/clinique-take-the-day-off-cleansing-balm-125-ml)\n\nBåda dessa produkter passar utmärkt för torr hud och hjälper till att bevara hudens naturliga fuktighet.
2,4,3,Jag har kombinerad hud och vill ha serum. Har du något tips?,kombinerad-hud,serum,Svaret ger bra rekommendationer för serum med hyaluronsyra för kombinerad hud och inkluderar produkter med pris och länkar. Det saknas dock en tydlig indikation på om dessa produkter är från den givna kontexten.,"För dig med kombinerad hud rekommenderar jag ""The Ordinary - Hyaluronic Acid 2% + B5"". Det är ett återfuktande fuktserum som fokuserar på att binda fukt utan att tillsätta olja, vilket är perfekt för att behålla balansen mellan de torra och oljiga partierna av din hud. Den vattenbaserade formulan känns lätt och icke-ocklusiv. Pris: 135 kr. Du kan hitta produkten här: [The Ordinary - Hyaluronic Acid 2% + B5](https://www.kicks.se/the-ordinary-hyaluronic-acid-2-b5-30-ml)\n\nEtt annat alternativ är ""BeautyAct - 5% Niacinamide Glass Serum"", som passar för att balansera och ge lyster till din hud. Detta serum innehåller niacinamid för att balansera huden och hyaluronsyra för att ge fukt, vilket skapar en ""glass skin""-effekt. Pris: 319 kr. Finns att köpa här: [BeautyAct - 5% Niacinamide Glass Serum](https://www.kicks.se/beautyact-5-niacinamide-glass-serum-30-ml)"


Medelbetyg: 4.33 / 5


Ovan test bekräftar att korrekt information hämtas. Även om utvärderingsmodell svarar att det är oklart om produktrekommendationen finns i kontexten, så har jag verifierat att det är korrekta produkter som hämtas, inkl länk och pris. Detta test hade kunnat förbättras med bättre formulering i facit, men jag känner mig trygg med att testet bekräftar att jag har en väl fungerande modell.

Jag är därmed nöjd med min Notebook och går vidare med att utveckla min Streamlit-app i en separat fil.

# REFLEKTION

## Affärsvärde
Denna RAG-lösning kan skapa affärsnytta genom att guida användare till rätt produkter baserat på deras behov. Som användare slipper man leta sig igenom hela Kicks sortiment och jämföra olika produkter, vilket sparar mycket tid. För en aktör som Kicks kan detta:

- Öka försäljning genom träffsäkra rekommendationer utifrån användarens input 
- Förbättra kundupplevelsen genom personlig rådgivning vilket skapar lojalitet
- Fungera som en “digital hudvårdsrådgivare” som är tillgänglig dygnet runt 

---

## Etik
Eftersom applikationen ger råd kring hudvård är det viktigt att:

- Inte ge medicinska eller vilseledande råd  
- Undvika att “hitta på” information utanför dokumenten  

En viktig aspekt är också att användaren inte bör se detta som en ersättning för professionell hudvårdsrådgivning eller dermatologisk expertis, så det har jag lagt in en text kring i min app.

Om bildanalys implementeras i framtiden, tillkommer ytterligare etiska frågor kring integritet och hantering av persondata.

---

## Utmaningar
Den största tekniska utmaningen i projektet har varit att balansera:

- En naturlig och flytande konversation (LLM)  
- Med krav på att svaren endast ska baseras på specifika dokument (RAG)  

Initialt gav modellen för generella svar, vilket löstes genom att:

- Justera system_prompt  
- Styra flödet via triggers och användarval (hudtyp och kategori)  

En annan utmaning var teknisk kompatibilitet:

- Problem uppstod vid användning av ChromaDB i Streamlit (det kraschade pga versionskonflikter hela tiden)
- Detta löstes genom att byta till FAISS, vilket fungerade stabilt i appen  

Projektet visade också vikten av att:

- Börja med en mindre scope (endast hudvård)  
- Iterera stegvis istället för att bygga allt på en gång  